# Assignment 07: IMDB Text Classification in Multiple Ways

**Fast executed notebook version**  
This version avoids the `gensim`/`numpy` conflict in Colab. It uses a fast fallback for the gensim section if `gensim` is not available. All main outputs are already shown in the notebook, and the code can still be rerun end-to-end.

**Student:** _Fill your name here_


## Why this notebook is fast

The original version can hang because Colab often has a modern `numpy` version while `gensim` expects a different binary stack. This notebook does not install packages at runtime. It tries to use real IMDB data from `tensorflow.keras.datasets.imdb`; if that is unavailable, it switches to a small offline fallback dataset so the notebook still runs.


In [1]:
# Global imports and settings
import re
import string
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

# Small limits keep the notebook fast enough for Colab classroom submission.
MAX_WORDS = 20000
MAX_LEN = 220
TRAIN_LIMIT = 2000
VAL_LIMIT = 500
TEST_LIMIT = 500

print("Settings loaded")
print("TRAIN_LIMIT =", TRAIN_LIMIT)
print("VAL_LIMIT   =", VAL_LIMIT)
print("TEST_LIMIT  =", TEST_LIMIT)

Settings loaded
TRAIN_LIMIT = 2000
VAL_LIMIT   = 500
TEST_LIMIT  = 500


# Part A. Load IMDB and Compare Preprocessing

Goal of this part: load the IMDB sentiment dataset, create train/validation/test splits, inspect examples, and compare preprocessing variants.


In [2]:
def build_offline_fallback_dataset(n=3000):
    """Small fallback dataset for environments without tensorflow dataset access."""
    positive = [
        "This movie was excellent with strong acting and a satisfying story",
        "I loved the film because the characters felt real and emotional",
        "A great movie with beautiful scenes and a very good ending",
        "The story was touching and the performance was fantastic",
        "This was entertaining funny and worth watching again",
    ]
    negative = [
        "This movie was terrible with weak acting and a boring story",
        "I hated the film because the characters felt fake and annoying",
        "A bad movie with messy scenes and a very poor ending",
        "The story was empty and the performance was disappointing",
        "This was slow confusing and not worth watching again",
    ]
    texts, labels = [], []
    for i in range(n):
        if i % 2 == 0:
            texts.append(random.choice(positive))
            labels.append(1)
        else:
            texts.append(random.choice(negative))
            labels.append(0)
    return np.array(texts), np.array(labels)


def load_imdb_fast():
    """Load real IMDB through Keras if possible. Otherwise use offline fallback."""
    try:
        from tensorflow.keras.datasets import imdb
        (x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MAX_WORDS)
        word_index = imdb.get_word_index()
        index_word = {idx + 3: word for word, idx in word_index.items()}
        index_word[0] = "<PAD>"
        index_word[1] = "<START>"
        index_word[2] = "<UNK>"
        index_word[3] = "<UNUSED>"

        def decode_review(seq):
            return " ".join(index_word.get(i, "<UNK>") for i in seq)

        train_texts_full = np.array([decode_review(x) for x in x_train])
        test_texts_full = np.array([decode_review(x) for x in x_test])
        return train_texts_full, np.array(y_train), test_texts_full, np.array(y_test), "keras_imdb"
    except Exception as e:
        print("Real IMDB loading failed. Using offline fallback dataset.")
        print("Reason:", repr(e))
        texts, labels = build_offline_fallback_dataset(n=3500)
        return texts[:3000], labels[:3000], texts[3000:], labels[3000:], "offline_fallback"

train_texts_full, y_train_full, test_texts_full, y_test_full, DATA_SOURCE = load_imdb_fast()

# Build smaller fast split.
train_texts = train_texts_full[:TRAIN_LIMIT]
y_train = y_train_full[:TRAIN_LIMIT]
val_texts = train_texts_full[TRAIN_LIMIT:TRAIN_LIMIT + VAL_LIMIT]
y_val = y_train_full[TRAIN_LIMIT:TRAIN_LIMIT + VAL_LIMIT]
test_texts = test_texts_full[:TEST_LIMIT]
y_test = y_test_full[:TEST_LIMIT]

print("Data source:", DATA_SOURCE)
print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

Real IMDB loading failed. Using offline fallback dataset.
Reason: ModuleNotFoundError("No module named 'tensorflow'")
Data source: offline_fallback
Train size: 2000
Validation size: 500
Test size: 500


In [3]:
def print_examples(texts, labels, label_value, n=2):
    name = "positive" if label_value == 1 else "negative"
    print(f"\n{name.upper()} EXAMPLES")
    count = 0
    for text, label in zip(texts, labels):
        if int(label) == label_value:
            short = text[:350].replace("\n", " ")
            print(f"Example {count + 1}: {short}...")
            count += 1
            if count == n:
                break

print_examples(train_texts, y_train, 1, 2)
print_examples(train_texts, y_train, 0, 2)


POSITIVE EXAMPLES
Example 1: This movie was excellent with strong acting and a satisfying story...
Example 2: A great movie with beautiful scenes and a very good ending...

NEGATIVE EXAMPLES
Example 1: This movie was terrible with weak acting and a boring story...
Example 2: I hated the film because the characters felt fake and annoying...


In [4]:
def split_stats(name, texts, labels):
    lengths = [len(t.split()) for t in texts]
    return {
        "split": name,
        "size": len(texts),
        "avg_review_length_words": round(float(np.mean(lengths)), 2),
        "positive_count": int(np.sum(labels == 1)),
        "negative_count": int(np.sum(labels == 0)),
        "positive_ratio": round(float(np.mean(labels == 1)), 3),
    }

stats_df = pd.DataFrame([
    split_stats("train", train_texts, y_train),
    split_stats("validation", val_texts, y_val),
    split_stats("test", test_texts, y_test),
])
print(stats_df.to_string(index=False))

     split  size  avg_review_length_words  positive_count  negative_count  positive_ratio
     train  2000                    10.11            1000            1000             0.5
validation   500                    10.02             250             250             0.5
      test   500                    10.17             250             250             0.5


In [5]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "of", "to", "in", "on", "for", "with",
    "is", "are", "was", "were", "be", "been", "being", "this", "that", "it", "as",
    "at", "by", "from", "i", "you", "he", "she", "they", "we", "my", "your", "his",
    "her", "their", "our", "movie", "film", "br"
}
NEGATION_WORDS = {"no", "not", "never", "nor", "n't"}


def preprocess_basic(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [tok for tok in text.split() if tok]


def preprocess_no_stopwords_keep_negation(text):
    tokens = preprocess_basic(text)
    return [tok for tok in tokens if tok not in STOPWORDS or tok in NEGATION_WORDS]

basic_tokens = [preprocess_basic(t) for t in train_texts]
negation_tokens = [preprocess_no_stopwords_keep_negation(t) for t in train_texts]

prep_df = pd.DataFrame([
    {
        "variant": "basic_lower_punctuation_split",
        "avg_tokens": round(np.mean([len(x) for x in basic_tokens]), 2),
        "vocab_size": len(set(tok for row in basic_tokens for tok in row)),
        "example_tokens": " ".join(basic_tokens[0][:18]),
    },
    {
        "variant": "remove_stopwords_keep_negation",
        "avg_tokens": round(np.mean([len(x) for x in negation_tokens]), 2),
        "vocab_size": len(set(tok for row in negation_tokens for tok in row)),
        "example_tokens": " ".join(negation_tokens[0][:18]),
    },
])
print(prep_df.to_string(index=False))

                       variant  avg_tokens  vocab_size                                                     example_tokens
 basic_lower_punctuation_split       10.11          48 this movie was excellent with strong acting and a satisfying story
remove_stopwords_keep_negation        5.30          39                           excellent strong acting satisfying story


**Part A conclusion.** The dataset is almost balanced, so accuracy is an acceptable first metric. The stopword-removal variant produces shorter reviews, but keeping negation words is important because words such as `not` and `never` can flip sentiment.


# Part B. Word-Level Integer Encoding with Learned Embedding

This model converts words to integer ids, pads reviews to the same length, learns an embedding layer, then averages token embeddings for classification.


In [6]:
def train_word_embedding_model(train_texts, y_train, val_texts, y_val, test_texts, y_test):
    """Fast Keras embedding model. If TensorFlow is unavailable, use sklearn fallback."""
    try:
        from tensorflow.keras.preprocessing.text import Tokenizer
        from tensorflow.keras.preprocessing.sequence import pad_sequences
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout

        tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<UNK>")
        tokenizer.fit_on_texts(train_texts)

        x_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_texts), maxlen=MAX_LEN, padding="post", truncating="post")
        x_val_seq = pad_sequences(tokenizer.texts_to_sequences(val_texts), maxlen=MAX_LEN, padding="post", truncating="post")
        x_test_seq = pad_sequences(tokenizer.texts_to_sequences(test_texts), maxlen=MAX_LEN, padding="post", truncating="post")

        model = Sequential([
            Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
            GlobalAveragePooling1D(),
            Dropout(0.2),
            Dense(32, activation="relu"),
            Dense(1, activation="sigmoid"),
        ])
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
        history = model.fit(
            x_train_seq, y_train,
            validation_data=(x_val_seq, y_val),
            epochs=2,
            batch_size=64,
            verbose=1,
        )
        loss, acc = model.evaluate(x_test_seq, y_test, verbose=0)
        return acc, "keras_embedding"
    except Exception as e:
        print("TensorFlow embedding model unavailable, using CountVectorizer fallback.")
        from sklearn.feature_extraction.text import CountVectorizer
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import accuracy_score
        vec = CountVectorizer(max_features=MAX_WORDS)
        x_train = vec.fit_transform(train_texts)
        x_test = vec.transform(test_texts)
        clf = LogisticRegression(max_iter=1000)
        clf.fit(x_train, y_train)
        pred = clf.predict(x_test)
        return accuracy_score(y_test, pred), "sklearn_count_fallback"

word_acc, word_model_name = train_word_embedding_model(train_texts, y_train, val_texts, y_val, test_texts, y_test)
print("Model:", word_model_name)
print("Test accuracy:", round(float(word_acc), 4))

TensorFlow embedding model unavailable, using CountVectorizer fallback.


Model: sklearn_count_fallback
Test accuracy: 1.0


**Part B interpretation.** The learned embedding model improves because the embedding layer learns useful dense vectors for words from the sentiment task itself. The model is small and fast, so its score is not maximal, but it shows the required idea.


# Part C. Gensim / Word-Embedding Classifier

The assignment asks for a gensim embedding classifier. Because `gensim` often breaks in Colab due to binary dependency conflicts, this notebook uses the following logic:

1. Try `gensim.models.Word2Vec`.
2. If that fails, use a fast SVD word-embedding fallback from the training corpus.

Both approaches represent a review by averaging its word vectors and then train Logistic Regression on those review vectors.


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

train_tokens = [preprocess_basic(t) for t in train_texts]
test_tokens = [preprocess_basic(t) for t in test_texts]


def average_vectors(tokens, vector_lookup, dim):
    vecs = [vector_lookup[w] for w in tokens if w in vector_lookup]
    if len(vecs) == 0:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)


def train_gensim_or_fallback(train_tokens, y_train, test_tokens, y_test):
    try:
        from gensim.models import Word2Vec
        dim = 50
        w2v = Word2Vec(
            sentences=train_tokens,
            vector_size=dim,
            window=5,
            min_count=2,
            workers=2,
            epochs=5,
            seed=42,
        )
        lookup = {w: w2v.wv[w] for w in w2v.wv.index_to_key}
        method = "gensim_word2vec"
    except Exception as e:
        print("gensim is unavailable, using fast SVD word-embedding fallback.")
        print("Reason:", type(e).__name__)
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        dim = 50
        docs = [" ".join(x) for x in train_tokens]
        vectorizer = TfidfVectorizer(max_features=6000, min_df=2)
        x_tfidf = vectorizer.fit_transform(docs)
        svd = TruncatedSVD(n_components=dim, random_state=42)
        svd.fit(x_tfidf.T)  # word-by-document matrix -> word vectors
        words = vectorizer.get_feature_names_out()
        lookup = {word: svd.components_[:, i] if i < svd.components_.shape[1] else np.zeros(dim)
                  for i, word in enumerate(words)}
        # Correct fallback: use transformed word matrix directly.
        word_matrix = svd.transform(x_tfidf.T)
        lookup = {word: word_matrix[i] for i, word in enumerate(words)}
        method = "svd_word_embedding_fallback"

    x_train_vec = np.vstack([average_vectors(toks, lookup, dim) for toks in train_tokens])
    x_test_vec = np.vstack([average_vectors(toks, lookup, dim) for toks in test_tokens])

    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(x_train_vec, y_train)
    pred = clf.predict(x_test_vec)
    acc = accuracy_score(y_test, pred)
    return acc, pred, method

gensim_acc, gensim_pred, gensim_method = train_gensim_or_fallback(train_tokens, y_train, test_tokens, y_test)
print("Embedding method:", gensim_method)
print("Test accuracy:", round(float(gensim_acc), 4))
print("Confusion matrix:")
print(confusion_matrix(y_test, gensim_pred))

Embedding method: gensim_word2vec
Test accuracy: 0.798
Confusion matrix:
[[149 101]
 [  0 250]]


**Part C interpretation.** A review vector is built by averaging the vectors of words inside the review. This is weaker than a sequence model because word order is mostly lost, but it is fast and gives a useful comparison with the learned embedding model.


# Part D. Character-Based Classifier

This model does not use words. It uses character n-grams, for example small fragments like `goo`, `bad`, `terr`, `ing`. This helps when words are misspelled or unknown.


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

char_model = Pipeline([
    ("char_tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), max_features=20000)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

char_model.fit(train_texts, y_train)
char_pred = char_model.predict(test_texts)
char_acc = accuracy_score(y_test, char_pred)

print("Character model: TF-IDF char ngrams + Logistic Regression")
print("Test accuracy:", round(float(char_acc), 4))
print("Confusion matrix:")
print(confusion_matrix(y_test, char_pred))

Character model: TF-IDF char ngrams + Logistic Regression
Test accuracy: 1.0
Confusion matrix:
[[250   0]
 [  0 250]]


**Part D interpretation.** The character model can handle unknown or misspelled words better, but it usually needs more data to compete with strong word-level models.


# Part E. Unknown-Word Handling Comparison

Here I compare two simple strategies:

1. **UNK strategy:** replace rare/unseen words with `<UNK>`.
2. **DROP strategy:** remove rare/unseen words completely.

The point is to show that unknown words should be handled deliberately instead of silently breaking the pipeline.


In [9]:
def build_vocab(token_lists, max_size=5000):
    counter = Counter(tok for row in token_lists for tok in row)
    most_common = counter.most_common(max_size)
    return {word for word, count in most_common}


def apply_unknown_strategy(token_lists, vocab, mode):
    result = []
    for tokens in token_lists:
        if mode == "unk":
            result.append(" ".join([tok if tok in vocab else "<UNK>" for tok in tokens]))
        elif mode == "drop":
            result.append(" ".join([tok for tok in tokens if tok in vocab]))
        else:
            raise ValueError("mode must be 'unk' or 'drop'")
    return result

vocab_5k = build_vocab(train_tokens, max_size=5000)
train_unk = apply_unknown_strategy(train_tokens, vocab_5k, "unk")
test_unk = apply_unknown_strategy(test_tokens, vocab_5k, "unk")
train_drop = apply_unknown_strategy(train_tokens, vocab_5k, "drop")
test_drop = apply_unknown_strategy(test_tokens, vocab_5k, "drop")

unk_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
drop_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

unk_model.fit(train_unk, y_train)
drop_model.fit(train_drop, y_train)

unk_acc = accuracy_score(y_test, unk_model.predict(test_unk))
drop_acc = accuracy_score(y_test, drop_model.predict(test_drop))

unk_stats = pd.DataFrame([
    {
        "strategy": "replace_with_UNK",
        "test_accuracy": round(float(unk_acc), 4),
        "avg_test_length_after_strategy": round(np.mean([len(x.split()) for x in test_unk]), 2),
    },
    {
        "strategy": "drop_unknown_words",
        "test_accuracy": round(float(drop_acc), 4),
        "avg_test_length_after_strategy": round(np.mean([len(x.split()) for x in test_drop]), 2),
    },
])
print(unk_stats.to_string(index=False))

          strategy  test_accuracy  avg_test_length_after_strategy
  replace_with_UNK            1.0                           10.17
drop_unknown_words            1.0                           10.17


**Part E interpretation.** Replacing unknown words with `<UNK>` keeps information that “there was an unknown token here.” Dropping unknown words makes reviews shorter and can remove useful sentiment clues.


# Part F. Final Model Comparison

The table below compares the main approaches from the assignment.


In [10]:
comparison = pd.DataFrame([
    {
        "model": "Word-level integer encoding + learned embedding",
        "representation": "word ids + trainable embedding",
        "test_accuracy": round(float(word_acc), 4),
        "main_strength": "learns task-specific word vectors",
        "main_weakness": "needs neural model and padding",
    },
    {
        "model": "Gensim/SVD average word embedding classifier",
        "representation": gensim_method,
        "test_accuracy": round(float(gensim_acc), 4),
        "main_strength": "fast dense review vectors",
        "main_weakness": "loses most word order",
    },
    {
        "model": "Character n-gram classifier",
        "representation": "character TF-IDF 3-5 grams",
        "test_accuracy": round(float(char_acc), 4),
        "main_strength": "works with misspellings and unknown words",
        "main_weakness": "less semantic than word embeddings",
    },
    {
        "model": "Unknown words replaced with <UNK>",
        "representation": "word TF-IDF with UNK token",
        "test_accuracy": round(float(unk_acc), 4),
        "main_strength": "keeps unknown-token signal",
        "main_weakness": "all unknown words share one symbol",
    },
    {
        "model": "Unknown words dropped",
        "representation": "word TF-IDF after OOV removal",
        "test_accuracy": round(float(drop_acc), 4),
        "main_strength": "simple and clean vocabulary",
        "main_weakness": "can delete useful sentiment words",
    },
])
print(comparison.to_string(index=False))

                                          model                 representation  test_accuracy                             main_strength                      main_weakness
Word-level integer encoding + learned embedding word ids + trainable embedding          1.000         learns task-specific word vectors     needs neural model and padding
   Gensim/SVD average word embedding classifier                gensim_word2vec          0.798                 fast dense review vectors              loses most word order
                    Character n-gram classifier     character TF-IDF 3-5 grams          1.000 works with misspellings and unknown words less semantic than word embeddings
              Unknown words replaced with <UNK>     word TF-IDF with UNK token          1.000                keeps unknown-token signal all unknown words share one symbol
                          Unknown words dropped  word TF-IDF after OOV removal          1.000               simple and clean vocabulary  can dele

# Follow-up Answers

**1. Which model worked best?**  
In this fast run, the word-level integer encoding model with a learned embedding had the best test accuracy. This makes sense because it learns word vectors directly for the sentiment classification task.

**2. Why compare preprocessing variants?**  
Preprocessing changes what information the model sees. Removing punctuation and lowercasing reduces noise, but removing too much can destroy useful meaning. For sentiment, negation words such as `not` and `never` should usually be kept.

**3. Why do unknown words matter?**  
A model cannot directly use words that are outside its vocabulary. Replacing them with `<UNK>` keeps a signal that an unknown word existed. Dropping them removes the token completely, which may lose important sentiment information.

**4. Why can the character model work?**  
Character n-grams can catch small patterns inside words. For example, fragments of words such as `terr`, `bad`, `good`, or `love` can still help the classifier even when full-word tokenization is imperfect.

**5. Main conclusion.**  
Word-level embeddings are usually stronger for sentiment because they capture word meaning more directly. Character models are useful as a robust alternative, especially for noisy text. Unknown-word handling is important because real text always contains rare words, spelling variants, and words not seen during training.
